# 第 8 周 - 笔记本 2：测试单个代理

## 目标
单独测试每个代理：
1. SpecialistAgent（在 Modal 上微调 Llama）
2.FrontierAgent（GPT-5.1 + RAG）
3. 比较结果

## 时间：15-20 分钟

In [ ]:
import sys
sys.path.append('..')

import os
from dotenv import load_dotenv
import chromadb

from src.agents import SpecialistAgent, FrontierAgent
from src.utils.items import Item
from src.utils.evaluator import evaluate
from src.config import config

# 负载环境
# Load environment
load_dotenv()

print("✅ Environment loaded")

## 加载测试数据

In [ ]:
print(f"Loading test data from: {config.DATASET_NAME}")
_, _, test = Item.from_hub(config.DATASET_NAME)

print(f"✅ Loaded {len(test):,} test items")

## 加载 ChromaDB 集合

In [ ]:
# 加载 ChromaDB
# Load ChromaDB
chroma_client = chromadb.PersistentClient(path="../data/chroma")
collection = chroma_client.get_collection(name="products")

print(f"✅ Loaded ChromaDB collection")
print(f"   Items in collection: {collection.count():,}")

## 测试 1：SpecialistAgent（微调 Llama）

**注意：** 这需要从第 7 周开始设置 Modal.com

In [ ]:
# 初始化 SpecialistAgent
# Initialize SpecialistAgent
specialist = SpecialistAgent()

print("✅ SpecialistAgent initialized")

In [ ]:
# 测试几个例子
# Test on a few examples
print("Testing SpecialistAgent on 3 products:\n")

for i in range(3):
    item = test[i]
    description = item.prompt or item.summary or item.title
    prediction = specialist.price(description)
    
    print(f"Product: {item.title[:50]}...")
    print(f"Actual: ${item.price:.2f}")
    print(f"Predicted: ${prediction:.2f}")
    print(f"Error: ${abs(prediction - item.price):.2f}\n")

In [ ]:
# 全面评价
# Full evaluation
def specialist_predict(item):
    description = item.prompt or item.summary or item.title
    return specialist.price(description)

print("Running full evaluation on SpecialistAgent...\n")
evaluate(specialist_predict, test, size=200, workers=1)

## 测试 2：FrontierAgent (GPT-5.1 + RAG)

In [ ]:
# 初始化FrontierAgent
# Initialize FrontierAgent
frontier = FrontierAgent(collection)

print("✅ FrontierAgent initialized")

In [ ]:
# 测试几个例子
# Test on a few examples
print("Testing FrontierAgent on 3 products:\n")

for i in range(3):
    item = test[i]
    description = item.prompt or item.summary or item.title
    prediction = frontier.price(description)
    
    print(f"Product: {item.title[:50]}...")
    print(f"Actual: ${item.price:.2f}")
    print(f"Predicted: ${prediction:.2f}")
    print(f"Error: ${abs(prediction - item.price):.2f}\n")

In [ ]:
# 全面评价
# Full evaluation
def frontier_predict(item):
    description = item.prompt or item.summary or item.title
    return frontier.price(description)

print("Running full evaluation on FrontierAgent...\n")
evaluate(frontier_predict, test, size=200, workers=1)

## 测试 3：置信度分数（新！）

In [ ]:
# 测试置信感知预测
# Test confidence-aware predictions
print("Testing confidence scores on 5 products:\n")

for i in range(5):
    item = test[i]
    description = item.prompt or item.summary or item.title
    
    # 充满信心地获得预测
    # Get predictions with confidence
    specialist_price, specialist_conf = specialist.price_with_confidence(description)
    frontier_price, frontier_conf = frontier.price_with_confidence(description)
    
    print(f"Product: {item.title[:50]}...")
    print(f"Actual: ${item.price:.2f}")
    print(f"\nSpecialist: ${specialist_price:.2f} (confidence: {specialist_conf:.2f})")
    print(f"Frontier: ${frontier_price:.2f} (confidence: {frontier_conf:.2f})")
    print("-" * 60 + "\n")

## 概括

✅ 个人代理测试完成！

**预期结果：**
- SpecialistAgent：~$40 错误（微调 Llama）
- FrontierAgent：大约 30 美元错误（GPT-5.1 + RAG）

**主要观察结果：**
1. FrontierAgent 更准确（RAG 有帮助！）
2. 置信度分数因产品而异
3. 有些产品比其他产品更容易预测

**下一步：** `03_ensemble.ipynb` - 将代理与置信权重结合起来